In [24]:
import cv2
import numpy as np

def preprocesar_imagen(ruta_imagen):
    """Carga la imagen y aplica filtros para reducir ruido."""
    img = cv2.imread(ruta_imagen)
    if img is None:
        return None, None
    
    # Conversión a escala de grises y desenfoque
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    suavizado = cv2.GaussianBlur(gris, (7, 7), 0)
    
    # Segmentación por umbral (Thresholding)
    _, binaria = cv2.threshold(suavizado, 120, 255, cv2.THRESH_BINARY_INV)
    
    # Limpieza morfológica para eliminar pequeñas imperfecciones
    kernel = np.ones((3, 3), np.uint8)
    limpia = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, kernel, iterations=2)
    
    return img, limpia

def extraer_puntos(imagen_binaria):
    """Detecta contornos y filtra aquellos que corresponden a los puntos del dado."""
    contornos, _ = cv2.findContours(imagen_binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    centros_puntos = []
    for cnt in contornos:
        area = cv2.contourArea(cnt)
        # Filtrado por área para ignorar ruido o el dado completo
        if 30 < area < 300:
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                centros_puntos.append((cx, cy))
                
    return centros_puntos

def identificar_dados(puntos, radio_agrupacion=80):
    """Agrupa los puntos detectados en estructuras individuales (dados)."""
    dados = []
    
    for p in puntos:
        asignado = False
        for grupo in dados:
            # Calcular el centroide actual del grupo para medir distancia
            centro_x = np.mean([pt[0] for pt in grupo])
            centro_y = np.mean([pt[1] for pt in grupo])
            
            distancia = np.linalg.norm(np.array(p) - np.array((centro_x, centro_y)))
            
            if distancia < radio_agrupacion:
                grupo.append(p)
                asignado = True
                break
        
        if not asignado:
            dados.append([p])
            
    return dados

def visualizar_analisis(original, grupos_dados):
    """Dibuja los resultados sobre la imagen original."""
    resultado = original.copy()
    
    for grupo in grupos_dados:
        valor = len(grupo)
        # Calcular ubicación del texto (promedio de los puntos del grupo)
        cx = int(np.mean([p[0] for p in grupo]))
        cy = int(np.mean([p[1] for p in grupo]))
        
        # Dibujar cada punto detectado
        for p in grupo:
            cv2.circle(resultado, p, 5, (0, 0, 255), -1)
            
        # Escribir el valor del dado
        cv2.putText(resultado, f"Valor: {valor}", (cx - 20, cy - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        
    return resultado

# --- Flujo Principal ---
ruta = "img/15.jpg"
img_original, img_binaria = preprocesar_imagen(ruta)

if img_original is not None:
    lista_puntos = extraer_puntos(img_binaria)
    dados_detectados = identificar_dados(lista_puntos)
    
    img_final = visualizar_analisis(img_original, dados_detectados)
    
    print(f"Dados encontrados: {len(dados_detectados)}")
    print(f"Valores: {[len(g) for g in dados_detectados]}")
    cv2.namedWindow("Deteccion de Dados", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Deteccion de Dados", 600, 400)

img_pequena = cv2.resize(img_final, None, fx=0.5, fy=0.5)
cv2.imshow("Deteccion de Dados", img_pequena)

cv2.waitKey(0)
cv2.destroyAllWindows()

Dados encontrados: 14
Valores: [11, 9, 4, 6, 4, 3, 1, 3, 3, 3, 1, 1, 1, 1]
